In [21]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SHEET_ID = "1GTcG_AdKIrmY9r1qkyomVcjUgpSSOHkkv-Zuj1lUwNM"
GID = "0"
url_tsv = f"https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=tsv&gid={GID}"

df = pd.read_csv(url_tsv, sep="\t", decimal=",")
print("Colunas lidas:", list(df.columns))
display(df.head())


FEATURE_SETS = {
    "BC":  ["Altura", "Peso"],              # colunas B,C
    "BD":  ["Altura", "Idade"],             # colunas B,D
    "CD":  ["Peso", "Idade"],            # colunas C,D
    "BCD": ["Altura", "Peso", "Idade"] # colunas B,C,D
}

FEATURES = FEATURE_SETS["BCD"]   # <-- troque aqui: "BC", "BD", "CD", "BCD"
TARGET = "Classe"            # coluna F (classe)
for col in FEATURES + ["Classe"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[TARGET] = df[TARGET].astype(str).str.strip().str.lower()
df = df.dropna(subset=FEATURES + [TARGET]).copy()
print("\nDistribuição do target:")
print(df[TARGET].value_counts())
le = LabelEncoder()
y_all = le.fit_transform(df[TARGET])
X_all = df[FEATURES].values
print("\nClasses:", list(le.classes_))
print("Total linhas válidas:", len(df))
print("Total de colunas:", len(df.columns))

RANDOM_STATE = 42
df_shuffled = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
test_df  = df_shuffled.iloc[:10].copy()
train_df = df_shuffled.iloc[10:30].copy()
X_train = train_df[FEATURES].values
y_train = le.transform(train_df[TARGET])
X_test  = test_df[FEATURES].values
y_test  = le.transform(test_df[TARGET])
print("\nTreino:", len(train_df), "| Teste:", len(test_df))
print("Atletas:", test_df)

K = 10
WEIGHTS = "distance"  # "uniform" ou "distance"
model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=K, weights=WEIGHTS))
])
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("\nFeatures usadas:", FEATURES)
print("Acurácia:", accuracy_score(y_test, y_pred))
print("\nRelatório:")
print(classification_report(y_test, y_pred, target_names=le.classes_))
print("Matriz de confusão (real x previsto):")
print(confusion_matrix(y_test, y_pred))

Colunas lidas: ['Altura', 'Peso', 'Idade', 'Classe']


,Altura,Peso,Idade,Classe
0,1.62,58,32,0
1,1.70,67,29,0
2,1.68,63,35,0
3,1.60,55,30,0
4,1.72,70,28,0



Distribuição do target:
Classe
0    18
1    12
Name: count, dtype: int64

Classes: ['0', '1']
Total linhas válidas: 30
Total de colunas: 4

Treino: 20 | Teste: 10
Atletas:    Altura  Peso  Idade Classe
0    1.86    91     23      1
1    1.74    74     27      0
2    1.90    95     21      1
3    1.75    76     28      0
4    1.63    59     34      0
5    1.71    68     30      0
6    1.84    88     24      1
7    1.83    87     26      1
8    1.65    61     32      0
9    1.62    58     32      0

Features usadas: ['Altura', 'Peso', 'Idade']
Acurácia: 0.9

Relatório:
              precision    recall  f1-score   support

           0       1.00      0.83      0.91         6
           1       0.80      1.00      0.89         4

    accuracy                           0.90        10
   macro avg       0.90      0.92      0.90        10
weighted avg       0.92      0.90      0.90        10

Matriz de confusão (real x previsto):
[[5 1]
 [0 4]]
